# Cabai menjelang Nataru: benar selalu melonjak?

Tiap akhir tahun ada kabar yang sama: harga cabai naik karena Nataru. Kabar itu
terdengar begitu pasti sehingga jarang ada yang menanyakannya lagi. Saya penasaran:
kalau disusun berdampingan tujuh musim Nataru terakhir, seberapa sering lonjakan itu
benar terjadi, untuk cabai jenis apa, dan apakah barang lain ikut-ikutan?

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))  # akar repo

import json
import numpy as np
import pandas as pd

from tools import gaya

nas = pd.read_csv("data/harga-nasional.csv", parse_dates=["tanggal"])
prov = pd.read_csv("data/harga-provinsi.csv", parse_dates=["tanggal"])

# musim = tahun Sep; rel = hari terhadap 25 Desember musim itu
for df in (nas, prov):
    df["musim"] = df["tanggal"].dt.year.where(df["tanggal"].dt.month >= 9, df["tanggal"].dt.year - 1)

KOMODITAS = ["Cabai Merah", "Cabai Rawit", "Bawang Merah", "Beras"]
MUSIM = sorted(nas["musim"].unique())


def rel_natal(t):
    jangkar = pd.Timestamp(t.year if t.month >= 9 else t.year - 1, 12, 25)
    return (t - jangkar).days


print("musim:", MUSIM)
print("rentang:", nas["tanggal"].min().date(), "s.d.", nas["tanggal"].max().date())
print("baris nasional:", len(nas), "| baris provinsi:", len(prov))

musim: [np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]
rentang: 2019-09-02 s.d. 2026-03-31
baris nasional: 4228 | baris provinsi: 141377


## Data & cara ukur

Datanya harga eceran harian dari PIHPPS (Pusat Informasi Harga Pangan Strategis
Nasional, bi.go.id/hargapangan), hasil survei enumerator di pasar tradisional yang
dikoordinasikan Bank Indonesia. Saya ambil empat komoditas: cabai merah dan cabai
rawit (dua komoditas yang jadi referensi kabar Nataru), plus bawang merah dan beras
sebagai pembanding. Rentangnya tujuh musim Nataru, September sampai Maret 2019/20
hingga 2025/26; hanya hari kerja yang tercatat.

Titik ukurnya harga nasional yang dihitung server sebagai rata-rata antar-provinsi,
jadi bukan rata-rata tertimbang jumlah penduduk. Per musim tersedia sekitar 151 hari
kerja per komoditas. Harga dasar tiap musim saya tetapkan sebagai median 1 September
sampai 20 November, masa tenang sebelum riak Nataru; yang diukur kemudian adalah
simpangan terhadap harga dasar itu. Panen dilakukan 22-23 September 2026 dari endpoint
snapshot harian (`GetGridData1`) dengan jeda sopan; respons mentah disimpan supaya
bisa ditelusuri ulang.

In [2]:
# cakupan dan pemeriksaan dasar
kak = nas.groupby(["komoditas", "musim"]).size().unstack(0)
print(kak)
print("\nhari berganda nasional (harus 0):", nas.duplicated(["tanggal", "komoditas"]).sum())
print("median tingkat harga per komoditas (Rp/kg):")
print(nas.groupby("komoditas")["harga_nasional"].median().round(0).astype(int))

komoditas  Bawang Merah  Beras  Cabai Merah  Cabai Rawit
musim                                                   
2019                151    151          151          151
2020                150    150          150          150
2021                150    150          150          150
2022                152    152          152          152
2023                151    151          151          151
2024                151    151          151          151
2025                152    152          152          152

hari berganda nasional (harus 0): 0
median tingkat harga per komoditas (Rp/kg):
komoditas
Bawang Merah    36750
Beras           12550
Cabai Merah     46100
Cabai Rawit     53350
Name: harga_nasional, dtype: int64


## Pembedahan

Langkah pertama: susun ketujuh musim di satu garis waktu, masing-masing diukur
sebagai persen simpangan dari harga dasarnya sendiri, lalu dihaluskan dengan rerata
bergerak lima hari supaya akhir pekan yang tak tercatat tidak mengganggu mata. Ini
untuk **cabai merah**, komoditas yang paling sering disebut dalam kabar Nataru.

In [3]:
JENDELA_DASAR = (-116, -35)   # 1 Sep s.d. 20 Nov: pra-Nataru
JENDELA_PUNCAK = (-15, 14)    # 10 Des s.d. 8 Jan: jendela lonjakan


def anomali_kom(kom):
    d = nas[nas["komoditas"] == kom].copy().set_index(["musim", "tanggal"])["harga_nasional"].unstack(0)
    per_musim = {}
    for m in d.columns:
        s = d[m].dropna()
        rel = s.index.to_series().map(rel_natal)
        dasar = s[rel.between(*JENDELA_DASAR)].median()       # median pra-Nataru musim itu
        per_musim[m] = pd.Series(((s / dasar - 1) * 100).values, index=rel.values)
    return pd.concat(per_musim, axis=1).sort_index()


ANOM = {k: anomali_kom(k) for k in KOMODITAS}
cm = ANOM["Cabai Merah"]
print("titik hari per musim:", cm.notna().sum().sum())

titik hari per musim: 1057


In [4]:
def grafik_1(mode):
    fig, ax, fs = gaya.dasar(
        mode,
        "Harga cabai merah: datar berbulan-bulan, lalu melompat saat Nataru",
        "Median puncak +50% (7 musim), tapi tidak otomatis: dua musim malah koreksi. "
        "Puncak jatuh kapan saja antara 11 Des dan 8 Jan.",
        "PIHPPS (BI), rata-rata nasional harga eceran pasar tradisional; 7 musim Nataru 2019/20-2025/26",
        int(cm.notna().sum().sum()),
    )
    cm5 = cm.rolling(5, min_periods=3, center=True).mean()
    for m in MUSIM:
        ax.plot(cm5.index, cm5[m], color=gaya.INK2, lw=1.0, alpha=0.45, zorder=2)
    med = cm5.median(axis=1)
    ax.plot(med.index, med.values, color=gaya.AKSEN, lw=2.2, zorder=3, label="median 7 musim")
    ax.axhline(0, color=gaya.GRID, lw=0.8)
    ax.axvline(0, color=gaya.AKSEN, lw=0.8, ls=":", zorder=1)
    ax.text(2, ax.get_ylim()[1] * 0.05, "25 Des", color=gaya.INK2, fontsize=8.5 * fs, va="bottom")
    bln = [(-116, "Sep"), (-86, "Okt"), (-55, "Nov"), (-24, "Des"), (6, "Jan"), (37, "Feb"), (65, "Mar")]
    ax.set_xticks([b for b, _ in bln])
    ax.set_xticklabels([n for _, n in bln])
    ax.set_ylabel("simpangan dari harga dasar pra-Nataru (%)", fontsize=9.5 * fs)
    ax.set_xlabel("garis tipis: tiap satu musim; garis tebal: median", fontsize=9.5 * fs)
    ax.grid(True, axis="y"); ax.grid(False, axis="x")
    ax.legend(frameon=False, fontsize=8.5 * fs, loc="upper left")
    gaya.simpan(fig, "01-anomali-musiman", mode)


for mode in gaya.MODE:
    grafik_1(mode)

In [5]:
# lonjakan puncak per musim per komoditas (untuk grafik 2 dan temuan)
def puncak_musim(anom):
    halus = anom.rolling(5, min_periods=3, center=True).mean()
    dalam = halus.loc[JENDELA_PUNCAK[0]:JENDELA_PUNCAK[1]]
    return dalam.max()


UP = pd.DataFrame({k: puncak_musim(a) for k, a in ANOM.items()})  # index musim
print(UP.round(1))
print("\nmedian:", UP.median().round(1).to_dict())

      Cabai Merah  Cabai Rawit  Bawang Merah  Beras
2019         -0.2        -14.4          51.5    0.6
2020         54.5        108.4          22.2    2.3
2021         50.5        112.8           2.7    0.9
2022        -16.7         20.0          10.8    4.5
2023         74.5         61.2          54.8    1.5
2024         66.7         53.0          39.9   -0.5
2025         12.1         58.4          26.4   -0.4

median: {'Cabai Merah': 50.5, 'Cabai Rawit': 58.4, 'Bawang Merah': 26.4, 'Beras': 0.9}


In [6]:
URUT = ["Beras", "Bawang Merah", "Cabai Merah", "Cabai Rawit"]

def grafik_2(mode):
    fig, ax, fs = gaya.dasar(
        mode,
        "Lonjakan Nataru: cabai paling jauh, beras nyaris tak ikut",
        "Median puncak: rawit +58%, merah +50%, bawang +26%, beras +1%. "
        "Dua titik merah di bawah nol adalah dua musim koreksi.",
        "PIHPPS (BI), puncak rolling 5 hari anomali harga nasional, jendela 10 Des-8 Jan, 7 musim",
        int(UP.notna().sum().sum()),
    )
    rng = np.random.default_rng(2026)
    for x, k in enumerate(URUT):
        nilai = UP[k].dropna()
        warna = gaya.AKSEN if k.startswith("Cabai") else gaya.INK2
        ax.scatter(x + rng.uniform(-0.13, 0.13, len(nilai)), nilai, s=26, color=warna,
                   alpha=0.75, zorder=3)
        med = nilai.median()
        ax.plot([x - 0.28, x + 0.28], [med, med], color=gaya.INK, lw=2.4, zorder=4)
        ax.text(x, nilai.max() + 4, f"median {med:+.0f}%", ha="center",
                fontsize=8.5 * fs, color=warna)
    ax.axhline(0, color=gaya.GRID, lw=0.8)
    ax.set_xticks(range(len(URUT)))
    ax.set_xticklabels(URUT)
    ax.set_ylabel("lonjakan puncak Nataru (% atas harga dasar)", fontsize=9.5 * fs)
    ax.set_xlabel("tiap titik satu musim; garis tebal median 7 musim", fontsize=9.5 * fs)
    ax.grid(True, axis="y"); ax.grid(False, axis="x")
    gaya.simpan(fig, "02-lonjakan-komoditas", mode)


for mode in gaya.MODE:
    grafik_2(mode)

## Temuan

> Iya, lonjakan Nataru itu nyata dan khas cabai: cabai rawit melonjak di 6 dari 7 musim
> (median puncak +58%), cabai merah di 5 dari 7 (median +50%), sementara beras praktis
> diam (+1%) dan bawang merah berada di antaranya (+26%). Tapi "selalu" itu rasa-rasanya
> terlalu pasti: harga cabai merah dua kali malah turun saat Nataru, dan harga yang sudah
> terlanjur naik jarang turun kembali sebelum Maret.

In [7]:
# metrik resmi temuan, semua dengan seed terkunci
from numpy.random import default_rng

rng = default_rng(2026)

def ci_median(x, n_boot=10000):
    x = np.asarray(x, float)
    sm = np.array([np.median(rng.choice(x, len(x))) for _ in range(n_boot)])
    return float(np.median(x)), float(np.percentile(sm, 2.5)), float(np.percentile(sm, 97.5))

metrik = {"komoditas": {}, "disparitas": {}, "volatilitas": {}, "plateau": {}}
for k in KOMODITAS:
    med, lo, hi = ci_median(UP[k].dropna().values)
    metrik["komoditas"][k] = {"median_uplift": med, "ci": [lo, hi],
                              "n_musim_positif": int((UP[k] > 0).sum())}

# disparitas provinsi saat hari puncak nasional cabai merah (per musim)
rasio_puncak = []
for m in MUSIM:
    s = nas[(nas["komoditas"] == "Cabai Merah") & (nas["musim"] == m)].sort_values("tanggal")
    rel = s["tanggal"].map(rel_natal)
    dasar = s.loc[rel.between(*JENDELA_DASAR), "harga_nasional"].median()
    an = pd.Series(((s["harga_nasional"] / dasar - 1) * 100).values, index=rel.values)
    h = an.rolling(5, min_periods=3, center=True).mean()
    dalam = h[(h.index >= JENDELA_PUNCAK[0]) & (h.index <= JENDELA_PUNCAK[1])]
    hari_puncak = (pd.Timestamp(f"{m}-12-25") + pd.Timedelta(days=int(dalam.idxmax())))
    nilai = prov[(prov["komoditas"] == "Cabai Merah") & (prov["musim"] == m)]
    selisih = (nilai["tanggal"] - hari_puncak).abs()
    hari = nilai.loc[selisih.idxmin(), "tanggal"]
    v = nilai[nilai["tanggal"] == hari]["harga"].dropna()
    rasio_puncak.append(float(v.max() / v.min()))
metrik["disparitas"]["rasio_provinsi_saat_puncak_median"] = float(np.median(rasio_puncak))
metrik["disparitas"]["rasio_semua_musim"] = [round(r, 2) for r in rasio_puncak]

# plateau: median anomali sisa musim (9 Jan-25 Mar) per musim
for k in ("Cabai Merah",):
    seg = ANOM[k].loc[15:90].median()
    metrik["plateau"][k] = {"per_musim": {str(m): round(float(seg[m]), 1) for m in MUSIM},
                            "median_across": float(seg.median()),
                            "n_positif": int((seg > 0).sum())}

# volatilitas: simpangan baku perubahan harian %, jendela ramai vs tenang
for k in KOMODITAS:
    d = nas[nas["komoditas"] == k].sort_values(["musim", "tanggal"]).copy()
    d["pct"] = d.groupby("musim")["harga_nasional"].pct_change() * 100
    rel = d["tanggal"].map(rel_natal)
    ramai = d.loc[rel.between(JENDELA_PUNCAK[0], JENDELA_PUNCAK[1]), "pct"].dropna()
    tenang = d.loc[rel.between(JENDELA_DASAR[0], JENDELA_DASAR[1]), "pct"].dropna()
    metrik["volatilitas"][k] = {"sd_ramai": float(ramai.std()), "sd_tenang": float(tenang.std()),
                                "rasio": float(ramai.std() / tenang.std())}

print(json.dumps(metrik, indent=2))
Path("data/metrik-005.json").write_text(json.dumps(metrik, indent=2))

{
  "komoditas": {
    "Cabai Merah": {
      "median_uplift": 50.47479912344777,
      "ci": [
        -0.15002885170225078,
        66.66666666666667
      ],
      "n_musim_positif": 5
    },
    "Cabai Rawit": {
      "median_uplift": 58.424110384894696,
      "ci": [
        19.95614035087719,
        108.40840840840843
      ],
      "n_musim_positif": 6
    },
    "Bawang Merah": {
      "median_uplift": 26.372315035799527,
      "ci": [
        10.815850815850814,
        51.49680457450387
      ],
      "n_musim_positif": 7
    },
    "Beras": {
      "median_uplift": 0.8583690987124415,
      "ci": [
        -0.4206098843322792,
        2.259887005649719
      ],
      "n_musim_positif": 5
    }
  },
  "disparitas": {
    "rasio_provinsi_saat_puncak_median": 3.7932773109243696,
    "rasio_semua_musim": [
      4.67,
      2.18,
      3.53,
      5.27,
      2.45,
      3.91,
      3.79
    ]
  },
  "volatilitas": {
    "Cabai Merah": {
      "sd_ramai": 6.4433196613852,
     

1780

## Batas & cara reproduksi

Lima hal perlu dibilang. Pertama, harga nasional adalah rata-rata antar-provinsi dari
server, bukan rata-rata tertimbang populasi; angkanya menceritakan "provinsi mewakili"
bukan "warga mewakili". Kedua, data hanya hari kerja, sehingga kurva dihaluskan lima
hari dan puncak tidak bisa dibaca per tanggal pasti. Ketiga, hanya tujuh musim: untuk
cabai merah selang kepercayaan median lonjakan masih lebar (menyentuh nol), jadi klaim
kuatnya berbunyi "5 dari 7 musim", bukan "selalu". Keempat, sisa musim setelah Januari
tumpang tindih dengan Ramadan di beberapa musim (paling jelas 2024/25, ketika Ramadan
jatuh Maret), jadi angka plateau-nya bukan murni warisan Nataru. Kelima, pilihan
tanggal menjadi sensitif terhadap lonjakan di luar jendela (misalnya gejolak bawang
merah 2019 yang jatuh tepat masa tenang musim lain tidak memengaruhi, karena dasar
dihitung per musim).

```bash
python3 tools/data_scraping.py     # panen snapshot harian PIHPPS (butuh internet, sekitar 2 jam)
python3 -m nbconvert --to notebook --execute --inplace analisis.ipynb
```